# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [1]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling
import datetime

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:60095/status,
Dashboard: http://127.0.0.1:60095/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:60096,Workers: 0
Dashboard: http://127.0.0.1:60095/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:49695,Total threads: 2
Dashboard: http://127.0.0.1:49697/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:60099,


# Value to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", 
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["Ihutai", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "LeftBank_Nov25", "ThePoint_Nov25"]



In [4]:
satellite_classes_all = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3, 'Gracilaria submerged': 4, 
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_all_unet = {'Seagrass': 0, 'Seagrass submerged': 1, 'Gracilaria': 2, 'Gracilaria submerged': 3, 
                           'Ulva': 4, 'Submerged vegetation': 5, 'Microphytobenthos': 6, 'Satmarsh': 7,
                           'Unvegetated': 8, 'Water': 9, 'Terrestrial': 10, 'Rock': 11, 'Mixed': 12,}

In [9]:
sample_method = "sampling_2"
method_2_threshold = .8
model_max_cloud_cover = 1 # percentage
model_low_tide_delta_hrs = 0
model_low_tide_delta_mins = 30
test_threshold = 0.1

predict_max_cloud_cover = 1 # percentage"
predict_low_tide_delta_hrs = 0
predict_low_tide_delta_mins = 30

tile_size=128
tile_stride=64
num_epoch=100
model_description = f"UN_model_all_trained_13_class_{int(test_threshold*100)}_percent_test_{num_epoch}_epochs_{tile_size}_tiling_{tile_stride}_stride"  
unet_type = "diffuser_unet_all_bands_all_SCL"
model_name = "diffuser-unet-epoch=60-val_loss=0.2856.ckpt"

'''unet_type = "diffuser_unet_6_bands"
model_name = "diffuser-unet-epoch=56-val_loss=0.4983.ckpt"'''

#model_name = "unetplusplus_test_on_10_percent_all_classes_100_epochs_64_tiling_None_stride/unet-epoch=58-val_loss=0.7639.ckpt"
#tile_size=64
#tile_stride=None

website_tab = (
    f"{model_description}_{unet_type}_train_{utils.get_samples_folder(sample_method, method_2_threshold)}_"
    f"{utils.get_low_tide_max_cloud_name(model_low_tide_delta_hrs, model_low_tide_delta_mins, model_max_cloud_cover)}_"
    f"predict_{utils.get_low_tide_max_cloud_name(predict_low_tide_delta_hrs, predict_low_tide_delta_mins, predict_max_cloud_cover)}"
)

In [10]:
model_path = utils.get_models_path(sample_method=sample_method,
                                   method_2_threshold=method_2_threshold,
                                   low_tide_delta_hrs=model_low_tide_delta_hrs,
                                   low_tide_delta_mins=model_low_tide_delta_mins,
                                   max_cloud_cover=model_max_cloud_cover)
model_path = model_path / model_description / unet_type
model_file = model_path / model_name
prediction_sites = large_seagrass_sites
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [11]:
data_path = utils.get_data_path()
utils.create_data_folders()
predictions_path, satellite_path = utils.get_prediction_path(
    sample_method, method_2_threshold,
    model_max_cloud_cover=model_max_cloud_cover,
    predict_max_cloud_cover=predict_max_cloud_cover,
    predict_low_tide_delta_hrs=predict_low_tide_delta_hrs,
    predict_low_tide_delta_mins=predict_low_tide_delta_mins,
    model_low_tide_delta_hrs=model_low_tide_delta_hrs,
    model_low_tide_delta_mins=model_low_tide_delta_mins)
predictions_path = predictions_path / model_description / unet_type
predictions_path.mkdir(parents=True, exist_ok=True)

# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

### Save satellite
The low-tide no cloud images across sites for a whole year

### Predict values

In [ ]:
for prediction_site in prediction_sites:

    website_output_folder = data_path / "website" / website_tab / prediction_site
    website_output_folder.mkdir(exist_ok=True, parents=True)
    if (website_output_folder / "info_all_dates.csv").exists():
        print(f"{prediction_site} already predicted and prepared for website. Skipping. Delete if unexpected.")
        continue

    ## Predict
    for year in years:
        print(f"{prediction_site}: year {year}")
        prediction_file = predictions_path / f"{prediction_site}_{year}.nc"
        polygon_file = utils.get_site_polygon_path(prediction_site)
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        
        if not prediction_file.exists():
            #model_feature_names_file = model_file.with_name(f"{model_file.stem}_feature_names.csv")
            if not satellite_file.exists():
                print("\tWARNING no satellite file to predict. Rerun the above cell if this is unexpected."
                      " Otherwise this indicates no suitable low-tide and low-cloud dates in the year.")
                continue
            predictions, satellite_data = training.predict_site_unet(
                test_satellite_file=satellite_file,
                polygon_file=polygon_file,
                checkpoint_file=model_file,
                tile_size=tile_size,
                stride=tile_stride)
            utils.save_netcdf(predictions, prediction_file)
        else:
            predictions = utils.load_classification(filename=prediction_file, chunks=None)
    
        for target_name in ["Seagrass", "Ulva", "Gracilaria"]:
            target_file = predictions_path /  f"{prediction_site}_{year}_{target_name.lower()}.gpkg"
            if not target_file.exists():
                target = {"date": [], "geometry": []}
                prediction = utils.load_classification(filename=prediction_file)
                prediction = utils.ensure_grid_indexing(prediction)
                target_id = pandas.read_csv(model_file.with_name(f"class_mappings.csv"), index_col=0)
                target_id = int(target_id.loc[target_id.index == target_name, "satellite_class_id"].squeeze())
                for time_index in range(len(prediction["time"])):
                    mask = predictions.isel(time=time_index) == target_id
                    target["date"].append(str(mask.time.data))
                    geometry = utils.mask_to_polygons(mask).dissolve().geometry
                    target["geometry"].append(geometry[0])
                target = geopandas.GeoDataFrame(target, crs="2193")
                target.to_file(target_file)

    ## Website
    website_options = {"value": prediction_sites, "label": prediction_sites}
    website_options = pandas.DataFrame(website_options)
    (data_path / "website" / website_tab).mkdir(exist_ok=True, parents=True)
    website_options.to_csv(data_path / "website" / website_tab / "options.csv", index=False)

    if (website_output_folder / "info_all_dates.csv").exists():
        print(f"{prediction_site} already run. Go to next")
        continue
    extents_all_years = {"Seagrass": [], "Ulva": [], "Gracilaria": []}
    info = {"ids": [], "B04_scale":[], "B03_scale":[], "B02_scale": [], "date":[]}
    for year in years:
        print(f"{prediction_site}: year {year}")
        satellite_info_set = False
        for target in extents_all_years.keys():
            target_file = predictions_path / f"{prediction_site}_{year}_{target.lower()}.gpkg"

            if not target_file.exists():
                print(f"WARNING no {target_file.name} file. Run 'download_satellite_predictions.ipynb' is this is unexpected.")
                continue
            target_extents = geopandas.read_file(target_file)
            extents_all_years[target].append(target_extents)

            for index, row in target_extents.iterrows():
                target_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
                date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
        
                filename = f"{date_YYMMDD}_{target.lower()}.gpkg"
                target_extent.to_file(website_output_folder / filename)

                # Look up the satellite tile information
                if not satellite_info_set:
                    #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
                    site_polygon = geopandas.read_file(utils.get_site_polygon_path(prediction_site))
                    tile_id, precentage_dict = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
                    
                    info["date"].append(row.date)
                    info["ids"].append(tile_id)
                    info["B04_scale"].append(precentage_dict["B04"])
                    info["B03_scale"].append(precentage_dict["B03"])
                    info["B02_scale"].append(precentage_dict["B02"]) 
            satellite_info_set = True
            
    info = pandas.DataFrame(info)
    info = info.rename(columns={"ids": "Satellite Tile IDs",
                                "B04_scale": "B04 Rescale",
                                "B03_scale": "B03 Rescale",
                                "B02_scale": "B02 Rescale"})
    
    for key, value in extents_all_years.items():
        print(f"\tTarget {key}")
        target_extents = pandas.concat(value, ignore_index=True)
        target_extents['area'] = target_extents.area
        info[f"{key} Area [m^2]"] = target_extents.area
        target_extents.dissolve()[["geometry"]].to_file(website_output_folder / f"{key.lower()}_presence_absence_map.gpkg")

    
    info.to_csv(website_output_folder / "info_all_dates.csv", index=False)
    

                

Ihutai: year 2016
	Loading U-Net checkpoint from C:\Local\repos\seagrass-detection\data\models\tide_30mins_cloud_1\sampling_2_80_percent\UN_model_all_trained_13_class_10_percent_test_100_epochs_128_tiling_64_stride\diffuser_unet_all_bands_all_SCL\diffuser-unet-epoch=60-val_loss=0.2856.ckpt
	Predict for 1 satellite images
	Combine predictions and write conventions
Ihutai: year 2017
	WARNING no satellite file to predict. Rerun the above cell if this is unexpected. Otherwise this indicates no suitable low-tide and low-cloud dates in the year.
Ihutai: year 2018
	Loading U-Net checkpoint from C:\Local\repos\seagrass-detection\data\models\tide_30mins_cloud_1\sampling_2_80_percent\UN_model_all_trained_13_class_10_percent_test_100_epochs_128_tiling_64_stride\diffuser_unet_all_bands_all_SCL\diffuser-unet-epoch=60-val_loss=0.2856.ckpt
	Predict for 2 satellite images
	Combine predictions and write conventions
Ihutai: year 2019
	Loading U-Net checkpoint from C:\Local\repos\seagrass-detection\data\

In [17]:
info

{'ids': [['S2A_MSIL2A_20160916T222802_R029_T59GPM_20210213T014056']],
 'percentages_2': [],
 'percentage_98': [],
 'date': ['2016-09-16 22:28:02.026000']}